# Chapter 02: Batches and Epochs

**Optional math reference:** [Mathematical view](00_math_intuition.ipynb), available for later. These programming lessons can be followed independently.

[Course index](../README.md) · [Previous](../01_training_foundations/README.md) · [Next](../03_validation_and_generalization/README.md)

**Goal:** Understand how batch size, example order, steps, and epochs change training.

**Study order:** read Chapter 01 first, then run this notebook from top to bottom.
This notebook contains its own setup and can run in a fresh kernel.

Before each experiment, record your prediction; afterwards, explain the output.
Related revision checkpoints: **5** in the
[revision notebook](../revision/01_training_basics_revision.ipynb).

## Contents

- [01. Inspect one example at a time: batch size 1](#lesson-01)
- [02. Reverse example order with batch size 1](#lesson-02)
- [03. Train with mini-batches of two examples](#lesson-03)
- [04. Compare batch sizes after one epoch](#lesson-04)
- [05. Compare epochs, updates, and examples processed](#lesson-05)


## Setup

Run this cell first in a fresh kernel. It imports PyTorch and recreates the original
four examples, so this chapter does not depend on another notebook's running kernel.

It also defines `one_step`, the manual-update reference from Chapter 01, for this
chapter's batch-size comparison. The helper uses local parameters and returns one update's results.


In [ ]:
import torch

# Original four examples; each x is paired with the y at the same position.
x = torch.tensor([0., 1., 2., 3.])
y = torch.tensor([2., 5., 8., 11.])
print("PyTorch version:", torch.__version__)
print("Inputs:", x.tolist())
print("Targets:", y.tolist())

# Reference helper from Chapter 01; used by lesson 04 below.
def one_step(lr):
    """Reset a model, apply one full-batch update, and return its results."""
    # A fresh start makes learning-rate comparisons reproducible.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)

    initial_loss = ((w * x + b - y) ** 2).mean()
    initial_loss.backward()

    with torch.no_grad():
        # lr scales the gradients; it does not calculate them.
        w -= lr * w.grad
        b -= lr * b.grad
        final_loss = ((w * x + b - y) ** 2).mean()

    return w.item(), b.item(), initial_loss.item(), final_loss.item()


## Lesson 01: Inspect one example at a time: batch size 1

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#batch-mean).

<a id="lesson-01"></a>



Reset and process inputs `0`, `1`, `2`, `3` in that order. Each example gets its own
loss, gradients, and update. The next example uses the newly learned parameters.
Four updates complete **one epoch**, because each of our four examples has been used once.

- At `x = 0`, prediction equals bias. Weight has no effect, so its gradient is zero.
- At `x = 1`, changing weight or bias by the same amount has the same effect on prediction;
  their gradients are equal.
- At `x = 2`, weight's gradient has twice the magnitude of bias's gradient.

Print gradients **before** the update and parameters **after** it. Measure mean loss
over the entire dataset at the end, so the reported loss covers all four examples.
Long decimals and `-0.0` are normal floating-point representations; `-0.0` is zero.


In [11]:
# Reset before the epoch; keep learned parameters between examples.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for i in range(len(x)):
    # The previous example's gradients must not carry into this example.
    w.grad = None
    b.grad = None

    # A batch of one contains one prediction and one squared error.
    prediction = w * x[i] + b
    loss = (prediction - y[i]) ** 2
    loss.backward()

    print(
        f"Input {x[i].item():.0f} | prediction={prediction.item():.4f} | "
        f"weight gradient={w.grad.item():.4f} | bias gradient={b.grad.item():.4f}"
    )

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(f"Step {i + 1} | weight={w.item():.4f} | bias={b.item():.4f}")

with torch.no_grad():
    # Evaluate the final parameters on ALL examples, not just the last one.
    total_loss = ((w * x + b - y) ** 2).mean()

forward_result = (w.item(), b.item(), total_loss.item())
print("Loss across all examples:", total_loss.item())


Input 0 | prediction=0.0000 | weight gradient=-0.0000 | bias gradient=-4.0000
Step 1 | weight=1.0000 | bias=0.2000
Input 1 | prediction=1.2000 | weight gradient=-7.6000 | bias gradient=-7.6000
Step 2 | weight=1.3800 | bias=0.5800
Input 2 | prediction=3.3400 | weight gradient=-18.6400 | bias gradient=-9.3200
Step 3 | weight=2.3120 | bias=1.0460
Input 3 | prediction=7.9820 | weight gradient=-18.1080 | bias gradient=-6.0360
Step 4 | weight=3.2174 | bias=1.3478
Loss across all examples: 0.16541965305805206


## Lesson 02: Reverse example order with batch size 1

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#order).

<a id="lesson-02"></a>



**Predict first:** will reversing the order change the final parameters? Keep the same
starting parameters and learning rate, and process inputs `3`, `2`, `1`, `0`.
Order matters because each update changes the parameters used for the next example.
Expect a final loss near `0.4012`, compared with `0.1654` in forward order.

With all examples in one batch, rearranging input-target pairs does not mathematically
change their mean loss or combined gradients at fixed parameters. Small numerical rounding
differences can occur. Keep each input paired with its own target when rearranging data.


In [12]:
# Reset so order is the only changed setting.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

# Iterate over positions 3, 2, 1, 0, keeping each x paired with its matching y.
for step, i in enumerate(range(len(x) - 1, -1, -1), start=1):
    w.grad = None
    b.grad = None
    loss = (w * x[i] + b - y[i]) ** 2
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(
        f"Step {step} | input={x[i].item():.0f} | "
        f"weight={w.item():.4f} | bias={b.item():.4f}"
    )

with torch.no_grad():
    total_loss = ((w * x + b - y) ** 2).mean()

print("Loss across all examples:", total_loss.item())


Step 1 | input=3 | weight=3.4000 | bias=0.8000
Step 2 | input=2 | weight=3.4800 | bias=0.8400
Step 3 | input=1 | weight=3.5480 | bias=0.9080
Step 4 | input=0 | weight=3.5480 | bias=1.0172
Loss across all examples: 0.40123653411865234


## Lesson 03: Train with mini-batches of two examples

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#batch-mean).

<a id="lesson-03"></a>



Reset and group inputs into `[0, 1]` and `[2, 3]`. Both examples in a batch use the same
current parameters. Average their squared errors, calculate gradients, and update **once**.
The second batch then uses the updated parameters. Two updates complete one epoch.

`range(0, len(x), 2)` gives starting positions `0` and `2`; `x[start:start + 2]`
selects the two consecutive inputs. `.mean()` averages errors; it does not sum them.
Expect final weight about `2.795`, bias `0.920`, and full-dataset loss `1.9777`.


In [13]:
# Reset to compare with the other batch sizes after one epoch.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for start in range(0, len(x), 2):
    # Select two inputs and their corresponding targets.
    batch_x = x[start:start + 2]
    batch_y = y[start:start + 2]

    w.grad = None
    b.grad = None

    # Combine both examples BEFORE calculating gradients and updating.
    predictions = w * batch_x + b
    loss = ((predictions - batch_y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(
        f"Inputs: {batch_x.tolist()} | "
        f"weight={w.item():.4f} | bias={b.item():.4f}"
    )

with torch.no_grad():
    # Use the same full-dataset metric as the batch-size-1 experiment.
    total_loss = ((w * x + b - y) ** 2).mean()

mini_batch_result = (w.item(), b.item(), total_loss.item())
print("Loss across all examples:", total_loss.item())


Inputs: [0.0, 1.0] | weight=1.2000 | bias=0.3000
Inputs: [2.0, 3.0] | weight=2.7950 | bias=0.9200
Loss across all examples: 1.977686882019043


## Lesson 04: Compare batch sizes after one epoch

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#counts).

<a id="lesson-04"></a>



Compare the saved forward-order results for batch sizes `1` and `2` with a fresh single
full-batch update. All runs start at weight `1`, bias `0`, with learning rate `0.05`.
Each processes four examples, but the update counts differ.

Smaller batches made more progress **in this experiment**, but also made more updates.
This does not establish that smaller batches always perform better. Batch size affects
both the examples contributing to each gradient and the frequency of parameter updates.
Run the setup cell and lessons 1 and 3 before this comparison cell so its function and saved results exist.


In [14]:
# Reuse one_step to get the full-batch result from the same initial parameters.
full_w, full_b, initial_loss, full_loss = one_step(0.05)

# Each row represents one epoch: each of the four examples is used once.
print("Batch size | Updates | Weight | Bias   | Full-dataset loss")
for batch_size, updates, result in [
    (4, 1, (full_w, full_b, full_loss)),
    (2, 2, mini_batch_result),
    (1, 4, forward_result),
]:
    final_w, final_b, final_loss = result
    print(
        f"{batch_size:10d} | {updates:7d} | "
        f"{final_w:.4f} | {final_b:.4f} | {final_loss:.4f}"
    )


Batch size | Updates | Weight | Bias   | Full-dataset loss
         4 |       1 | 2.0000 | 0.5000 | 10.2500
         2 |       2 | 2.7950 | 0.9200 | 1.9777
         1 |       4 | 3.2174 | 1.3478 | 0.1654


## Lesson 05: Compare epochs, updates, and examples processed

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#counts).

<a id="lesson-05"></a>


For this four-example dataset:

| Batch size | Updates per epoch | Epochs needed for four updates | Examples processed over those epochs |
|---|---:|---:|---:|
| 1 | 4 | 1 | 4 |
| 2 | 2 | 2 | 8 |
| 4 | 1 | 4 | 16 |

Batch size `4` needs **four epochs** to match the four updates made by batch size `1`
in one epoch. But it processes sixteen examples in total: the same four examples
revisited four times. Equal epochs, equal updates, and equal examples processed are
different ways to compare training runs. State which budget you are holding fixed.

### What we learned

- Weight is a kind of parameter; bias is another parameter.
- Loss measures error; gradients describe how parameter changes affect that loss.
- `backward()` calculates gradients; the update code changes parameters.
- Clearing `.grad` clears stored gradients while preserving learned parameters.
- The learning rate scales gradients into updates. A fixed rate does not imply fixed-sized updates.
- A batch contributes to one update. An epoch covers the whole dataset once.
- With batch size 1, each update changes the starting point for the next example, so order matters.
- Every loss compared in Chapters 01 and 02 is measured on the same four training examples.
  Chapter 03 evaluates unseen examples; overfitting is a later lesson.
